## 0. Proof of lifeRun this first. It prints immediately, so a blank log means the run hasnot started — not that it is stuck. It also tells you whether Internetis on, which is off by default on Kaggle and breaks every install.

In [ ]:
# Immediate proof of life. Kaggle's first log lines are debugger noise; this is# the first thing that is actually yours, so it prints before anything slow.import sys, platform, subprocess, timeSTART = time.time()print("=" * 58, flush=True)print(f"  notebook started  {time.strftime('%Y-%m-%d %H:%M:%S')}", flush=True)print(f"  python {sys.version.split()[0]} on {platform.platform()}", flush=True)try:    n = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],                       capture_output=True, text=True, timeout=20).stdout.strip()    print(f"  gpu: {n or 'none (fine for harvesting)'}", flush=True)except Exception:    print("  gpu: none (fine for harvesting)", flush=True)print(f"  internet: ", end="", flush=True)try:    import urllib.request    urllib.request.urlopen("https://pypi.org", timeout=15)    print("ON", flush=True)except Exception as e:    print(f"OFF or blocked -- {type(e).__name__}. "          "Settings > Internet > On (needs a phone-verified account).", flush=True)print("=" * 58, flush=True)

# Train YOLO26 on FRC fuel — ColabRuns the **training** half only. Keep the harvesting pipeline on your ownmachine: YouTube routinely blocks yt-dlp from datacenter IPs with"Sign in to confirm you're not a bot", and Colab is a datacenter IP.**Runtime → Change runtime type → GPU** before you start.

## 1. Check what GPU you gotFree tier is usually a T4. Anything is a large step up from an M2.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!pip -q install "ultralytics>=8.4"import torch, ultralyticsprint("ultralytics", ultralytics.__version__, "| torch", torch.__version__,      "| cuda", torch.cuda.is_available())print("  pip install finished", flush=True)

## 2. Get the dataset inBuild a **portable** archive locally first — `prepare_dataset.py` symlinks bydefault, and a plain `tar` of symlinks arrives empty:```bash.venv-train/bin/python train/prepare_dataset.py --copy --clean.venv-train/bin/python train/autolabel_fuel.pytar -czf dataset.tgz dataset/```Then either upload `dataset.tgz` to Drive (survives disconnects — recommended)or straight into the session with the file picker.

In [ ]:
from google.colab import drivedrive.mount('/content/drive')!tar -xzf /content/drive/MyDrive/dataset.tgz -C /content/!ls /content/dataset/images/train | wc -l

### dataset.yaml paths must be rewrittenThe file records an absolute path from your Mac, which does not exist here.

In [ ]:
from pathlib import Pathy = Path('/content/dataset/dataset.yaml')lines = [l for l in y.read_text().splitlines() if not l.startswith('path:')]y.write_text('path: /content/dataset\n' + '\n'.join(lines) + '\n')print(y.read_text())

## 3. Train`batch=-1` is Ultralytics' AutoBatch: it probes the card and sizes the batch toroughly 60% of VRAM. Do not hardcode it. The P2 head at imgsz=1280 ismemory-hungry — a stride-4 feature map is sixteen times the area of stride-16 —and a fixed `batch=8` needs something like 24 GB, which OOMs a 16 GB T4.`imgsz` is the setting that actually matters for this dataset. Fuel is ~17x12px, so resolution is what makes it detectable at all; `yolo26s-p2` adds astride-4 head for the same reason.

In [ ]:
from ultralytics import YOLOmodel = YOLO('yolo26s-p2.yaml').load('yolo26s.pt')model.train(    data='/content/dataset/dataset.yaml',    imgsz=1280, epochs=100, batch=-1,   # AutoBatch: sizes to ~60% of VRAM device=0,    project='/content/runs', name='fuel26',    scale=0.25, mosaic=1.0, close_mosaic=15,    fliplr=0.5, flipud=0.0, degrees=0.0, patience=30,)

## 4. Save the weights somewhere that survivesColab wipes local disk on disconnect. Copy to Drive **before** the session ends.

In [ ]:
!mkdir -p /content/drive/MyDrive/frc_weights!cp /content/runs/fuel26/weights/best.pt /content/drive/MyDrive/frc_weights/!cp /content/runs/fuel26/results.csv     /content/drive/MyDrive/frc_weights/!ls -la /content/drive/MyDrive/frc_weights/

## 5. Sanity-check before you trust it

In [ ]:
import globfrom ultralytics import YOLObest = YOLO('/content/runs/fuel26/weights/best.pt')best.predict(sorted(glob.glob('/content/dataset/images/train/*.jpg'))[:4],             imgsz=1280, save=True, project='/content/pred', name='check')from IPython.display import Image, displayfor p in sorted(glob.glob('/content/pred/check/*.jpg')):    display(Image(p, width=1100))

## Then, back on the Mac```bash# put best.pt somewhere sensible, then.venv-train/bin/yolo predict model=weights/best.pt source=data/frames/ imgsz=1280```### Free-tier realities- Sessions die after ~12 h, sooner if idle — Drive checkpoints matter.- GPU access is not guaranteed and can be throttled after heavy use.- `patience=30` stops early once val mAP plateaus, which usually beats  grinding all 100 epochs.